In [1]:
import pandas as pd
import numpy as np
import os
import re

DATA_DIR = "datasets"  # folder with your 39 CSVs
files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
print(f"Found {len(files)} files")

# First pass: just look at shape and columns of every file
inventory = []
for f in files:
    path = os.path.join(DATA_DIR, f)
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin1")
    try:
        inventory.append({
            "file": f,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "columns": list(df.columns)
        })
    except Exception as e:
        print(f"Failed to read {f}: {e}")

inv_df = pd.DataFrame(inventory)
pd.set_option('display.max_colwidth', None)
print(inv_df[["file", "rows", "cols"]])

Found 40 files
                                   file  rows  cols
0          RS_Session_254_AU_1496.A.csv     7     3
1        RS_Session_254_AU_2384.C.i.csv    27     4
2           RS_Session_254_AU_697_1.csv    37     3
3          RS_Session_255_AU_2349_2.csv    35     3
4           RS_Session_255_AU_305.A.csv     2     4
5           RS_Session_255_AU_749.C.csv     3     3
6           RS_Session_256_AS_154.A.csv     3     3
7          RS_Session_256_AU_2673_1.csv    16     3
8      RS_Session_256_AU_2673_2.ii_.csv    26     4
9            RS_Session_256_AU_95_C.csv    33    11
10       RS_Session_257_AS_71_A.ii_.csv    14     3
11         RS_Session_258_AU_2037_A.csv     4     4
12          RS_Session_258_AU_429_1.csv    35    18
13          RS_Session_258_AU_429_B.csv     4     3
14           RS_Session_258_AU_91_1.csv    35    18
15         RS_Session_259_AU_2028_A.csv     4     3
16         RS_Session_259_AU_2032_A.csv    11     2
17         RS_Session_259_AU_2474_A.csv     3    

In [2]:
def clean_column_names(df):
    """Standardize headers: strip whitespace, fix typos, lowercase-friendly."""
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={"hoi": "Sl. No."})  # fixes the typo in one file
    return df

def clean_numeric_column(series):
    """Convert messy numeric-looking columns to float, handling NA/approx/commas."""
    return (
        series.astype(str)
        .str.replace("approx.", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"NA": np.nan, "N/A": np.nan, "-": np.nan, "": np.nan})
        .astype(float)
    )

def load_and_clean(filename):
    path = os.path.join(DATA_DIR, filename)
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin1")
    df = clean_column_names(df)
    return df

In [3]:
theme_groups = {
    "ev_registration_trends": [
        "RS_Session_255_AU_749.C.csv",
        "RS_Session_256_AS_154.A.csv",
        "RS_Session_259_AU_2032_A.csv",
        "RS_Session_259_AU_2477_C_and_D.csv",
        "RS_Session_260_AU_1872_A_and_B.csv",
        "RS_Session_263_AU_905_A.csv",
        "RS_Session_265_AU_1355_A_and_B.csv",
    ],
    "state_wise_ev_data": [
        "RS_Session_254_AU_697_1.csv",
        "RS_Session_255_AU_2349_2.csv",
        "RS_Session_256_AU_95_C.csv",
        "RS_Session_258_AU_429_1.csv",
        "RS_Session_258_AU_91_1.csv",
        "RS_Session_259_AU_3475_1.csv",
        "RS_Session_260_AU_2349_A_to_B.csv",
    ],
    "charging_infrastructure": [
        "RS_Session_256_AU_2673_1.csv",
        "RS_Session_256_AU_2673_2.ii_.csv",
        "RS_Session_257_AS_71_A.ii_.csv",
        "RS_Session_259_AU_2474_A.csv",
        "RS_Session_259_AU_2837_A.csv",
        "RS_Session_259_AU_2837_B.csv",
        "RS_Session_265_AU_2151_E.csv",
        "RS_Session_266_AU_2960_B_i.csv",
        "RS_Session_266_AU_2960_C_to_D_i.csv",
        "RS_Session_267_AU_581_C_i.csv",
    ],
    "scheme_budget_and_subsidy": [
        "RS_Session_254_AU_1496.A.csv",
        "RS_Session_258_AU_2037_A.csv",
        "RS_Session_258_AU_429_B.csv",
        "RS_Session_259_AU_2028_A.csv",
        "RS_Session_259_AU_2836_A.csv",
        "RS_Session_259_AU_3477_A_to_D.csv",
        "RS_Session_263_AU_105_A.csv",
        "RS_Session_266_AS_217_4.csv",
        "RS_Session_266_AU_553_A_to_B.csv",
        "RS_Session_267_AU_580_A.csv",
    ],
    "sales_by_category": [
        "RS_Session_263_AU_102_A.csv",
        "RS_Session_263_AU_105_C.csv",
        "RS_Session_260_AS_241_E.csv",
    ],
    "global_comparison": [
        "RS_Session_266_AU_552_D_i.csv",
    ],
    "manufacturer_specific": [
        "RS_Session_255_AU_305.A.csv",
        "RS_Session_254_AU_2384.C.i.csv",
    ],
}

# quick check: did every file get assigned to a theme?
assigned = set(sum(theme_groups.values(), []))
unassigned = set(files) - assigned
print("Unassigned files:", unassigned if unassigned else "None — all files grouped")

Unassigned files: None — all files grouped


In [4]:
def build_registration_trends():
    frames = []
    for f in theme_groups["ev_registration_trends"]:
        df = load_and_clean(f)
        df["source_file"] = f
        frames.append(df)
        print(f"\n{f}:\n{df.head(2)}")  # inspect before deciding how to align columns
    return frames

frames = build_registration_trends()


RS_Session_255_AU_749.C.csv:
  Sl. No.  Year  Number of Electric Vehicles                  source_file
0       1  2019                       161314  RS_Session_255_AU_749.C.csv
1       2  2020                       119648  RS_Session_255_AU_749.C.csv

RS_Session_256_AS_154.A.csv:
   Year  Number of Vehicles  Percentage Change WRT Previous Year  \
0  2019              164852                                  NaN   
1  2020              123528                               -25.07   

                   source_file  
0  RS_Session_256_AS_154.A.csv  
1  RS_Session_256_AS_154.A.csv  

RS_Session_259_AU_2032_A.csv:
   Year  Total Count                   source_file
0  2014         2391  RS_Session_259_AU_2032_A.csv
1  2015         7790  RS_Session_259_AU_2032_A.csv

RS_Session_259_AU_2477_C_and_D.csv:
   Year  Electric Vehicles Registered in a Calendar Year  \
0  2020                                           124026   
1  2021                                           329808   

   Percentag

In [5]:
# --- EV Registration Trends: build simple year-level table ---
simple_files = [
    "RS_Session_255_AU_749.C.csv",
    "RS_Session_256_AS_154.A.csv",
    "RS_Session_259_AU_2032_A.csv",
    "RS_Session_259_AU_2477_C_and_D.csv",
    "RS_Session_263_AU_905_A.csv",
    "RS_Session_265_AU_1355_A_and_B.csv",
]

rename_maps = {
    "RS_Session_255_AU_749.C.csv": {"Year": "year", "Number of Electric Vehicles": "ev_count"},
    "RS_Session_256_AS_154.A.csv": {"Year": "year", "Number of Vehicles": "ev_count",
                                     "Percentage Change WRT Previous Year": "pct_change"},
    "RS_Session_259_AU_2032_A.csv": {"Year": "year", "Total Count": "ev_count"},
    "RS_Session_259_AU_2477_C_and_D.csv": {
        "Year": "year",
        "Electric Vehicles Registered in a Calendar Year": "ev_count",
        "Percentage Increase in registration from the previous year": "pct_change"},
    "RS_Session_263_AU_905_A.csv": {"Calendar Year": "year", "Electric Vehicles Registered": "ev_count"},
    "RS_Session_265_AU_1355_A_and_B.csv": {"Year": "year", "Number of Registered Electric Vehicles": "ev_count"},
}

frames = []
for f in simple_files:
    df = load_and_clean(f)
    df = df.rename(columns=rename_maps[f])
    df["source_file"] = f
    df = df[[c for c in ["year", "ev_count", "pct_change", "source_file"] if c in df.columns]]
    frames.append(df)

master_ev_registration_trends = pd.concat(frames, ignore_index=True, sort=False)
master_ev_registration_trends.info()
master_ev_registration_trends

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   year         30 non-null     object 
 1   ev_count     30 non-null     int64  
 2   source_file  30 non-null     object 
 3   pct_change   4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.1+ KB


,year,ev_count,source_file,pct_change
0,2019,161314,RS_Session_255_AU_749.C.csv,NaN
1,2020,119648,RS_Session_255_AU_749.C.csv,NaN
2,Total,280962,RS_Session_255_AU_749.C.csv,NaN
3,2019,164852,RS_Session_256_AS_154.A.csv,NaN
4,2020,123528,RS_Session_256_AS_154.A.csv,-25.07
5,2021,324840,RS_Session_256_AS_154.A.csv,162.97
6,2014,2391,RS_Session_259_AU_2032_A.csv,NaN
7,2015,7790,RS_Session_259_AU_2032_A.csv,NaN
8,2016,49622,RS_Session_259_AU_2032_A.csv,NaN
9,2017,86720,RS_Session_259_AU_2032_A.csv,NaN


In [6]:
master_ev_registration_trends = master_ev_registration_trends[
    ~master_ev_registration_trends["year"].astype(str).str.contains("Total", case=False, na=False)
].reset_index(drop=True)

master_ev_registration_trends.info()
master_ev_registration_trends

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   year         27 non-null     object 
 1   ev_count     27 non-null     int64  
 2   source_file  27 non-null     object 
 3   pct_change   4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 996.0+ bytes


,year,ev_count,source_file,pct_change
0,2019,161314,RS_Session_255_AU_749.C.csv,NaN
1,2020,119648,RS_Session_255_AU_749.C.csv,NaN
2,2019,164852,RS_Session_256_AS_154.A.csv,NaN
3,2020,123528,RS_Session_256_AS_154.A.csv,-25.07
4,2021,324840,RS_Session_256_AS_154.A.csv,162.97
5,2014,2391,RS_Session_259_AU_2032_A.csv,NaN
6,2015,7790,RS_Session_259_AU_2032_A.csv,NaN
7,2016,49622,RS_Session_259_AU_2032_A.csv,NaN
8,2017,86720,RS_Session_259_AU_2032_A.csv,NaN
9,2018,129125,RS_Session_259_AU_2032_A.csv,NaN


In [7]:
# --- EV Registration Trends: category-wise wide table → long format ---
df_wide = load_and_clean("RS_Session_260_AU_1872_A_and_B.csv")

records = []
years = ["2018", "2019", "2020", "2021", "2022", "2023 (Till 01-08-2023)"]
for _, row in df_wide.iterrows():
    for yr in years:
        records.append({
            "year": yr,
            "vehicle_category": row["Vehicle Category"],
            "total_vehicles": row.get(f"{yr} - Total"),
            "ev_count": row.get(f"{yr} - EV"),
            "ev_pct": row.get(f"{yr} - %"),
            "source_file": "RS_Session_260_AU_1872_A_and_B.csv"
        })

master_ev_registration_by_category = pd.DataFrame(records)
master_ev_registration_by_category.info()
master_ev_registration_by_category

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              30 non-null     object 
 1   vehicle_category  30 non-null     object 
 2   total_vehicles    30 non-null     int64  
 3   ev_count          30 non-null     int64  
 4   ev_pct            30 non-null     float64
 5   source_file       30 non-null     object 
dtypes: float64(1), int64(2), object(3)
memory usage: 1.5+ KB


,year,vehicle_category,total_vehicles,ev_count,ev_pct,source_file
0,2018,Two Wheeler,19576235,17067,0.09,RS_Session_260_AU_1872_A_and_B.csv
1,2019,Two Wheeler,18644700,30389,0.16,RS_Session_260_AU_1872_A_and_B.csv
2,2020,Two Wheeler,14305129,29113,0.20,RS_Session_260_AU_1872_A_and_B.csv
3,2021,Two Wheeler,13926217,156243,1.12,RS_Session_260_AU_1872_A_and_B.csv
4,2022,Two Wheeler,15592118,631181,4.05,RS_Session_260_AU_1872_A_and_B.csv
5,2023 (Till 01-08-2023),Two Wheeler,9276337,489637,5.28,RS_Session_260_AU_1872_A_and_B.csv
6,2018,Three Wheeler,764806,110133,14.40,RS_Session_260_AU_1872_A_and_B.csv
7,2019,Three Wheeler,765867,133489,17.43,RS_Session_260_AU_1872_A_and_B.csv
8,2020,Three Wheeler,400893,90385,22.55,RS_Session_260_AU_1872_A_and_B.csv
9,2021,Three Wheeler,390820,158129,40.46,RS_Session_260_AU_1872_A_and_B.csv


In [8]:
for f in theme_groups["state_wise_ev_data"]:
    df = load_and_clean(f)
    print(f"\n{f} — shape {df.shape}")
    display(df.head(2))


RS_Session_254_AU_697_1.csv — shape (37, 3)


,Sl. No.,State/UT,Total Number of Invoice/Sales
0,1,Jammu Kashmir,437
1,2,Himachal Pradesh,241



RS_Session_255_AU_2349_2.csv — shape (35, 3)


,Sl. No.,State/UT,Total Number of Invoices/Sales
0,1,Jammu Kashmir,1036
1,2,Himachal Pradesh,446



RS_Session_256_AU_95_C.csv — shape (33, 11)


...

,Sr. No.,State/UT Name,2WN,2WT,2WIC,3WN,3WT,LMV,LPV,LGV,4WIC,MMV,MPV,MGV,HPV,HGV,OTH,Grand Total
0,1,Andaman and Nicobar Island,2,5.0,NaN,NaN,30.0,86,6.0,NaN,NaN,NaN,NaN,NaN,40.0,NaN,NaN,169
1,2,Andhra Pradesh,27629,NaN,2.0,374.0,108.0,1050,3.0,166.0,NaN,NaN,NaN,NaN,NaN,NaN,1117.0,30449



RS_Session_259_AU_3475_1.csv — shape (35, 3)


,S. No.,State Name,Electric Vehicle Count
0,1,Andaman and Nicobar Island,182
1,2,Andhra Pradesh,51322



RS_Session_260_AU_2349_A_to_B.csv — shape (35, 4)


,Sl.No.,State/UT,Electric,Non-electric
0,1,Andaman and Nicobar Islands,190,161258
1,2,Andhra Pradesh,67905,16553509


In [9]:
for f in theme_groups["state_wise_ev_data"]:
    df = load_and_clean(f)
    print(f"\n{f} — shape {df.shape}")
    display(df.head(2))


RS_Session_254_AU_697_1.csv — shape (37, 3)


,Sl. No.,State/UT,Total Number of Invoice/Sales
0,1,Jammu Kashmir,437
1,2,Himachal Pradesh,241



RS_Session_255_AU_2349_2.csv — shape (35, 3)


,Sl. No.,State/UT,Total Number of Invoices/Sales
0,1,Jammu Kashmir,1036
1,2,Himachal Pradesh,446



RS_Session_256_AU_95_C.csv — shape (33, 11)


...

,Sr. No.,State/UT Name,2WN,2WT,2WIC,3WN,3WT,LMV,LPV,LGV,4WIC,MMV,MPV,MGV,HPV,HGV,OTH,Grand Total
0,1,Andaman and Nicobar Island,2,5.0,NaN,NaN,30.0,86,6.0,NaN,NaN,NaN,NaN,NaN,40.0,NaN,NaN,169
1,2,Andhra Pradesh,27629,NaN,2.0,374.0,108.0,1050,3.0,166.0,NaN,NaN,NaN,NaN,NaN,NaN,1117.0,30449



RS_Session_259_AU_3475_1.csv — shape (35, 3)


,S. No.,State Name,Electric Vehicle Count
0,1,Andaman and Nicobar Island,182
1,2,Andhra Pradesh,51322



RS_Session_260_AU_2349_A_to_B.csv — shape (35, 4)


,Sl.No.,State/UT,Electric,Non-electric
0,1,Andaman and Nicobar Islands,190,161258
1,2,Andhra Pradesh,67905,16553509


In [10]:
df_429 = load_and_clean("RS_Session_258_AU_429_1.csv")
df_91 = load_and_clean("RS_Session_258_AU_91_1.csv")
print(df_429.drop(columns=["Sl.No." if "Sl.No." in df_429.columns else "S.No."], errors="ignore").equals(
    df_91.drop(columns=["Sr. No."], errors="ignore")
))

False


In [11]:
df_429 = load_and_clean("RS_Session_258_AU_429_1.csv")
df_91 = load_and_clean("RS_Session_258_AU_91_1.csv")

# align state name columns so we can compare row by row
state_col_429 = "State Name"
state_col_91 = "State/UT Name"

df_429_sorted = df_429.sort_values(state_col_429).reset_index(drop=True)
df_91_sorted = df_91.sort_values(state_col_91).reset_index(drop=True)

print("Shapes:", df_429_sorted.shape, df_91_sorted.shape)
print("\nStates in 429 but not in 91:",
      set(df_429_sorted[state_col_429]) - set(df_91_sorted[state_col_91]))
print("States in 91 but not in 429:",
      set(df_91_sorted[state_col_91]) - set(df_429_sorted[state_col_429]))

# compare Grand Total column state by state
compare = df_429_sorted[[state_col_429, "Grand Total"]].merge(
    df_91_sorted[[state_col_91, "Grand Total"]],
    left_on=state_col_429, right_on=state_col_91,
    suffixes=("_429", "_91")
)
compare["diff"] = compare["Grand Total_429"] - compare["Grand Total_91"]
compare[compare["diff"] != 0]

Shapes: (35, 18) (35, 18)

States in 429 but not in 91: set()
States in 91 but not in 429: set()


,State Name,Grand Total_429,State/UT Name,Grand Total_91,diff


In [12]:
# --- Group 3: vehicle type-code breakdown by state (melt wide -> long) ---
# Note: RS_Session_258_AU_429_1.csv is a duplicate of this file (verified identical
# Grand Totals for all 35 states) — excluded to avoid double-counting.

df_type = load_and_clean("RS_Session_258_AU_91_1.csv")
df_type = df_type[~df_type["State/UT Name"].astype(str).str.contains("Total", case=False, na=False)]

type_cols = [c for c in df_type.columns if c not in
             ["Sr. No.", "State/UT Name", "Grand Total", "source_file"]]

master_state_vehicle_type = df_type.melt(
    id_vars=["State/UT Name"],
    value_vars=type_cols,
    var_name="vehicle_type_code",
    value_name="count"
).rename(columns={"State/UT Name": "state"})
master_state_vehicle_type["source_file"] = "RS_Session_258_AU_91_1.csv"

master_state_vehicle_type.info()
master_state_vehicle_type

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   state              510 non-null    object 
 1   vehicle_type_code  510 non-null    object 
 2   count              276 non-null    float64
 3   source_file        510 non-null    object 
dtypes: float64(1), object(3)
memory usage: 16.1+ KB


,state,vehicle_type_code,count,source_file
0,Andaman and Nicobar Island,2WN,2.0,RS_Session_258_AU_91_1.csv
1,Andhra Pradesh,2WN,27629.0,RS_Session_258_AU_91_1.csv
2,Arunachal Pradesh,2WN,14.0,RS_Session_258_AU_91_1.csv
3,Assam,2WN,2287.0,RS_Session_258_AU_91_1.csv
4,Bihar,2WN,13472.0,RS_Session_258_AU_91_1.csv
...,...,...,...,...
505,Tripura,OTH,NaN,RS_Session_258_AU_91_1.csv
506,Dadra and Nagar Haveli and Daman and Diu,OTH,4.0,RS_Session_258_AU_91_1.csv
507,Uttar Pradesh,OTH,1.0,RS_Session_258_AU_91_1.csv
508,Uttarakhand,OTH,NaN,RS_Session_258_AU_91_1.csv
